# UNIST ↔ 울산역 운영 최적화 에이전트 (Operation Optimizer)

v4 예약 노트북과 **같은 Google Sheets**를 읽어, 누적된 예약·운행 실적으로
**슬롯 등급 재편성 / N\* 재학습 / 운행시각 점검**을 자동 제안하는 에이전트.

```
입력  : ① 예약 시트(v4)  ② 운행실적 시트(신규)  ③ 셔틀 시간표 상수
처리  : 1) 슬롯 등급 재편성  2) N*·편익/인 재학습  3) 운행시각 점검
출력  : 재조정 제안서(자연어 + 표) → 사람 승인 → 다음 주 반영
```

**핵심 설계**
- v4와 100% 호환: 예약 시트 헤더 `name, direction, ktx_time, travel_date, created_at` 그대로 사용.
- **예약자=탑승자 근사**(`RIDERS_FROM_RESERVATIONS=True`): 운행실적 시트가 없어도 v4 예약 데이터만으로 작동. 예약 수를 탑승 수로 간주(노쇼율만 보정).
- 운행실적 시트(`UNIST_shuttle_operations`)는 선택: 실제 탑승 인원을 입력하면 근사 대신 실측을 사용.
- **데이터가 모두 부족하면** 자동으로 데모 데이터를 생성해 작동 시연(`USE_DEMO_DATA=True`).
- 모든 판정은 **결정론적 코드**가 계산하고, GPT는 그 결과를 **사람이 읽을 제안서로 서술**만 함(숫자 창작 금지).

> ⚠️ 제안서는 **자동 적용이 아니라** 사람(학교 행정) 승인을 전제로 한 권고임.


In [ ]:
# [셀 1] 설치
!pip install -q openai gspread gradio

In [ ]:
# [셀 2] 임포트 & 설정
import json
from collections import defaultdict
from dataclasses import dataclass
from datetime import datetime, date, timedelta

# ── 운영 정책 파라미터 (전부 조정 가능) ──────────────────────────────
PROMOTE_WEEKS   = 3   # 조건부 → 고정 승격: 연속 N주 예약 >= 임계치
DEMOTE_WEEKS    = 3   # 고정 → 조건부 강등: 연속 N주 평균탑승 < 강등 기준
RETIRE_WEEKS    = 4   # 조건부 → 미운영 폐지: 연속 N주 예약 < 폐지 기준

CONDITIONAL_MIN = 8   # 조건부 배차 임계치 N* (요금 2,000원 기준, 레포 optimization.py와 동일)
DEMOTE_LOAD     = 8   # 고정 강등 기준: 평균 탑승 < N*(손익분기 8명) 지속 시
RETIRE_RESV     = 3   # 폐지 기준: 예약 < 3명

CAPACITY        = 12  # 셔틀 정원
OP_COST_KRW     = 35000  # 1회 왕복 운영비 (레포 optimization.py: C=35,000)

# ── N* 재학습용 편익 파라미터 (레포 optimization.py와 동일) ──────────
P_TAXI    = 0.30     # 셔틀 이용자 중 기존 택시 비율
P_BUS     = 0.70     # 셔틀 이용자 중 기존 버스 비율
TAXI_FARE = 12000    # 택시 요금
BUS_FARE  = 1500     # 513 요금
T_SAVED_H = 20/60    # 버스 대비 절약시간(시간)
VOT       = 10000    # 학생 시간가치(원/h)
POLICY_FARE = 2000   # 확정 정책 요금 (→ N*=8)

# ── 데이터 소스 모드 ────────────────────────────────────────────────
# 예약자=탑승자 근사: 운행실적 시트가 없어도 v4 예약 데이터만으로 작동.
#   True  → 예약 수를 실제 탑승 수로 간주 (노쇼는 NO_SHOW_RATE로 보정)
#   False → 운행실적 시트(actual_riders)를 사용 (실측이 쌓인 경우)
RIDERS_FROM_RESERVATIONS = True
NO_SHOW_RATE = 0.1     # 예약 대비 노쇼 추정 비율 (예약자=탑승자 근사 시 탑승 = 예약×(1-이 값))

# ── 데모 모드 ───────────────────────────────────────────────────────
USE_DEMO_DATA = True   # 예약·운행 데이터가 모두 부족하면 데모로 시연. 실전 전환 시 False.
DEMO_WEEKS    = 6      # 데모로 생성할 과거 주차 수

print("설정 로드 완료")

In [ ]:
# [셀 3] Google Sheets 연결 — 예약 시트(v4) + 운행실적 시트(신규)
# ※ 최초 실행 시 구글 계정 인증 팝업이 뜹니다.
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# --- 예약 시트 (v4가 쓰는 것과 동일 이름) ---
RESV_SHEET   = "UNIST_shuttle_reservations"
RESV_HEADER  = ["name", "direction", "ktx_time", "travel_date", "created_at"]

# --- 운행실적 시트 (신규) ---
OPS_SHEET    = "UNIST_shuttle_operations"
OPS_HEADER   = ["op_date", "direction", "ktx_time", "slot", "service",
                "reservations", "actual_riders", "dispatched", "note"]

def _open_or_create(name, header):
    try:
        sh = gc.open(name)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(name)
    ws = sh.sheet1
    vals = ws.get_all_values()
    if not vals or vals[0] != header:
        ws.clear()
        ws.append_row(header, value_input_option="RAW")
    return sh, ws

resv_sh, resv_ws = _open_or_create(RESV_SHEET, RESV_HEADER)
ops_sh,  ops_ws  = _open_or_create(OPS_SHEET,  OPS_HEADER)

print("예약 시트:", resv_sh.url)
print("운행실적 시트:", ops_sh.url)

In [ ]:
# [셀 4] 셔틀 시간표 상수 (v4 셀 6과 동일 — 등급 판정의 기준표)
WEEKDAY_KR = "월화수목금토일"

SHUTTLE_FIXED = {
    "to_station": [
        {"slot": "금 오후", "wd": 4, "ktx": "13:58", "shuttle": "13:41"},
        {"slot": "금 저녁", "wd": 4, "ktx": "17:51", "shuttle": "17:34"},
        {"slot": "금 야간", "wd": 4, "ktx": "22:39", "shuttle": "22:22"},
        {"slot": "토 오전", "wd": 5, "ktx": "10:02", "shuttle": "09:45"},
    ],
    "to_campus": [
        {"slot": "월 오전", "wd": 0, "ktx": "08:45", "shuttle": "09:00"},
        {"slot": "일 오후", "wd": 6, "ktx": "12:07", "shuttle": "12:22"},
        {"slot": "일 저녁", "wd": 6, "ktx": "18:43", "shuttle": "18:58"},
        {"slot": "일 야간", "wd": 6, "ktx": "20:29", "shuttle": "20:44"},
    ],
}
SHUTTLE_CONDITIONAL = {
    "to_station": [
        {"slot": "목 오후", "wd": 3, "ktx": "13:58", "shuttle": "13:41"},
        {"slot": "목 저녁", "wd": 3, "ktx": "17:51", "shuttle": "17:34"},
        {"slot": "목 야간", "wd": 3, "ktx": "22:39", "shuttle": "22:22"},
        {"slot": "금 오전", "wd": 4, "ktx": "10:02", "shuttle": "09:45"},
        {"slot": "토 오후", "wd": 5, "ktx": "13:58", "shuttle": "13:41"},
    ],
    "to_campus": [],
}

def slot_key(direction, wd, ktx):
    """슬롯 고유 키."""
    return f"{direction}|{wd}|{ktx}"

# 기준표를 평탄화: key -> {grade, slot, direction, wd, ktx, shuttle}
SLOT_CATALOG = {}
for grade, table in [("fixed", SHUTTLE_FIXED), ("conditional", SHUTTLE_CONDITIONAL)]:
    for d, entries in table.items():
        for e in entries:
            SLOT_CATALOG[slot_key(d, e["wd"], e["ktx"])] = {
                "grade": grade, "slot": e["slot"], "direction": d,
                "wd": e["wd"], "ktx": e["ktx"], "shuttle": e["shuttle"],
            }
print(f"기준표 슬롯 {len(SLOT_CATALOG)}개 로드 (고정 8 + 조건부 5)")

In [ ]:
# [셀 5] 데이터 로드 — 실데이터 우선, 부족하면 데모 생성
import random

def _iso_week(d):
    """date -> (ISO year, ISO week) 튜플. '주차' 식별자."""
    y, w, _ = d.isocalendar()
    return (y, w)

def load_reservations():
    """예약 시트 -> 주차별·슬롯별 예약 수 집계."""
    rows = resv_ws.get_all_records()
    agg = defaultdict(int)   # (week, slotkey) -> count
    for r in rows:
        td = str(r.get("travel_date", "")).strip()
        ktx = str(r.get("ktx_time", "")).strip()
        d   = str(r.get("direction", "")).strip()
        if not (td and ktx and d):
            continue
        try:
            dt = datetime.strptime(td, "%Y-%m-%d").date()
        except ValueError:
            continue
        key = slot_key(d, dt.weekday(), ktx)
        if key in SLOT_CATALOG:
            agg[(_iso_week(dt), key)] += 1
    return agg

def load_operations():
    """운행실적 시트 -> 주차별·슬롯별 (예약, 실제탑승, 배차여부)."""
    rows = ops_ws.get_all_records()
    agg = {}  # (week, slotkey) -> dict
    for r in rows:
        od  = str(r.get("op_date", "")).strip()
        ktx = str(r.get("ktx_time", "")).strip()
        d   = str(r.get("direction", "")).strip()
        if not (od and ktx and d):
            continue
        try:
            dt = datetime.strptime(od, "%Y-%m-%d").date()
        except ValueError:
            continue
        key = slot_key(d, dt.weekday(), ktx)
        if key not in SLOT_CATALOG:
            continue
        agg[(_iso_week(dt), key)] = {
            "reservations": int(r.get("reservations", 0) or 0),
            "actual_riders": int(r.get("actual_riders", 0) or 0),
            "dispatched": str(r.get("dispatched", "")).strip().upper() in ("TRUE", "1", "Y", "YES"),
        }
    return agg

def make_demo():
    """데모용 6주치 가짜 데이터. 발표 시연용 — 실데이터 아님을 명시할 것.
    의도적으로 '목 오후'를 매주 임계치 초과(→승격 후보),
    '금 야간' 고정을 저조(→강등 후보)하게 설계."""
    random.seed(7)
    today = date.today()
    resv = defaultdict(int)
    ops  = {}
    for w in range(DEMO_WEEKS, 0, -1):
        ref = today - timedelta(weeks=w)
        wk  = _iso_week(ref)
        for key, meta in SLOT_CATALOG.items():
            slot = meta["slot"]
            if slot == "목 오후":              # 승격 후보: 매주 8~10명
                resvN = random.randint(8, 10)
            elif slot == "금 야간":            # 강등 후보(고정): 2~4명
                resvN = random.randint(2, 4)
            elif meta["grade"] == "conditional":
                resvN = random.randint(2, 6)   # 임계치 근처에서 흔들림
            else:
                resvN = random.randint(6, 12)  # 일반 고정: 양호
            resv[(wk, key)] = resvN
            dispatched = (meta["grade"] == "fixed") or (resvN >= CONDITIONAL_MIN)
            riders = min(resvN, CAPACITY) if dispatched else 0
            # 노쇼 약간 반영
            riders = max(0, riders - random.randint(0, 1)) if dispatched else 0
            ops[(wk, key)] = {"reservations": resvN, "actual_riders": riders,
                              "dispatched": dispatched}
    return resv, ops

resv_agg = load_reservations()
ops_agg  = load_operations()

DATA_IS_DEMO = False

# 예약자=탑승자 근사: 운행실적 시트가 비어 있으면 예약 집계로 ops_agg를 합성
if RIDERS_FROM_RESERVATIONS and len(ops_agg) < len(SLOT_CATALOG):
    synth = {}
    for (wk, key), resvN in resv_agg.items():
        meta = SLOT_CATALOG.get(key)
        if meta is None:
            continue
        # 고정편은 항상 배차 / 조건부편은 예약 >= 임계치일 때만 배차된다고 가정
        dispatched = (meta["grade"] == "fixed") or (resvN >= CONDITIONAL_MIN)
        riders = round(resvN * (1 - NO_SHOW_RATE)) if dispatched else 0
        synth[(wk, key)] = {"reservations": resvN, "actual_riders": riders,
                            "dispatched": dispatched}
    if synth:                 # 예약 데이터가 한 건이라도 있으면 그것으로 진행
        ops_agg = synth

# 그래도 데이터가 없으면(예약도 비었으면) 데모로 시연
if USE_DEMO_DATA and (len(ops_agg) < len(SLOT_CATALOG)):
    resv_agg, ops_agg = make_demo()
    DATA_IS_DEMO = True

# 주차 목록(정렬)
ALL_WEEKS = sorted({wk for (wk, _key) in ops_agg.keys()})

if DATA_IS_DEMO:
    src = "⚠️ 데모 데이터 사용 중 (실데이터 아님)"
elif RIDERS_FROM_RESERVATIONS:
    src = f"✅ v4 예약 데이터 사용 (예약자=탑승자 근사, 노쇼 {int(NO_SHOW_RATE*100)}% 보정)"
else:
    src = "✅ 운행실적 시트 사용 (실측 탑승)"
print(src, f"| 주차 {len(ALL_WEEKS)}개, 슬롯 {len(SLOT_CATALOG)}개")

In [ ]:
# [셀 6] 핵심 판정 엔진 (결정론적) — 등급 재편성 / N* 재학습 / 시각 점검
import math

def _recent_streak(key, predicate):
    """가장 최근 주부터 연속으로 predicate(weekdata)=True인 주 수."""
    streak = 0
    for wk in reversed(ALL_WEEKS):
        wd = ops_agg.get((wk, key))
        if wd is None:
            break
        if predicate(wd):
            streak += 1
        else:
            break
    return streak

def reclassify():
    """각 슬롯의 승격/강등/폐지/유지 판정."""
    proposals = []
    for key, meta in SLOT_CATALOG.items():
        grade = meta["grade"]
        weeks_data = [ops_agg[(wk, key)] for wk in ALL_WEEKS if (wk, key) in ops_agg]
        if not weeks_data:
            continue
        avg_resv  = sum(w["reservations"] for w in weeks_data) / len(weeks_data)
        avg_rider = sum(w["actual_riders"] for w in weeks_data) / len(weeks_data)

        action, reason = "유지", ""
        if grade == "conditional":
            promote_streak = _recent_streak(key, lambda w: w["reservations"] >= CONDITIONAL_MIN)
            retire_streak  = _recent_streak(key, lambda w: w["reservations"] <  RETIRE_RESV)
            if promote_streak >= PROMOTE_WEEKS:
                action = "고정 승격"
                reason = f"{promote_streak}주 연속 예약 ≥ {CONDITIONAL_MIN}명 (평균 {avg_resv:.1f}명)"
            elif retire_streak >= RETIRE_WEEKS:
                action = "미운영 폐지"
                reason = f"{retire_streak}주 연속 예약 < {RETIRE_RESV}명 (평균 {avg_resv:.1f}명)"
            else:
                reason = f"승격까지 연속 {promote_streak}/{PROMOTE_WEEKS}주 (평균 예약 {avg_resv:.1f}명)"
        elif grade == "fixed":
            demote_streak = _recent_streak(key, lambda w: w["actual_riders"] < DEMOTE_LOAD)
            if demote_streak >= DEMOTE_WEEKS:
                action = "조건부 강등"
                reason = f"{demote_streak}주 연속 평균탑승 < {DEMOTE_LOAD}명 (평균 {avg_rider:.1f}명)"
            else:
                reason = f"평균 탑승 {avg_rider:.1f}명 (강등 기준 {DEMOTE_LOAD}명 이상 유지)"

        proposals.append({
            "slot": meta["slot"], "direction": meta["direction"], "grade": grade,
            "ktx": meta["ktx"], "shuttle": meta["shuttle"],
            "avg_resv": round(avg_resv, 1), "avg_rider": round(avg_rider, 1),
            "action": action, "reason": reason,
        })
    # 변경 제안을 위로 정렬
    order = {"고정 승격": 0, "조건부 강등": 1, "미운영 폐지": 2, "유지": 3}
    proposals.sort(key=lambda p: (order.get(p["action"], 9), -p["avg_resv"]))
    return proposals

def relearn_threshold():
    """실측 탑승 분포로 편익/인과 N*를 재계산.
    공식은 레포 optimization.py와 동일 (요금은 택시·버스 양쪽에서 차감).
    (실측 요금수용 데이터가 없으면 초기 가정값 사용 — 구조만 시연.)"""
    fare = POLICY_FARE
    taxi = P_TAXI * (TAXI_FARE - fare)
    bus  = P_BUS  * ((T_SAVED_H * VOT) + BUS_FARE - fare)
    benefit = taxi + bus
    nstar = math.ceil(OP_COST_KRW / benefit)
    return {"benefit_per_person": round(benefit), "nstar": nstar, "fare": fare,
            "note": f"요금 {fare:,}원 기준. 실측 요금수용·시간절약 수집 시 P_TAXI·VOT를 갱신."}

proposals = reclassify()
threshold = relearn_threshold()

print("== 등급 재편성 판정 ==")
for p in proposals:
    mark = "→ " + p["action"] if p["action"] != "유지" else "   유지"
    print(f'{p["slot"]:6s} [{p["grade"]:11s}] 예약{p["avg_resv"]:4.1f} 탑승{p["avg_rider"]:4.1f}  {mark}  ({p["reason"]})')
print("\n== N* 재학습 ==")
print(threshold)

In [ ]:
# [셀 7] GPT 제안서 작성 — 계산 결과를 사람이 읽을 자연어 리포트로 (숫자 창작 금지)
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
MODEL  = "gpt-4o-mini"

REPORT_SYSTEM = """너는 UNIST 셔틀 운영 분석가다. 아래 JSON은 코드가 계산한 운영 데이터다.
이 숫자만 근거로, 학교 행정 담당자에게 보낼 '주간 운영 재조정 제안서'를 한국어로 작성하라.

[규칙]
- JSON에 없는 수치/시각은 절대 만들지 마라.
- 구조: (1) 한 줄 요약  (2) 변경 제안(승격/강등/폐지) 각 항목에 근거 수치 명시  (3) N* 재학습 결과  (4) 마지막에 '본 제안은 사람 승인 후 적용' 명시.
- 제안이 없으면 '이번 주 변경 권고 없음'이라고 분명히 써라.
- 담백하고 실무적인 톤. 과장 금지."""

def build_report():
    if DATA_IS_DEMO:
        data_source = "데모 데이터(실데이터 아님)"
    elif RIDERS_FROM_RESERVATIONS:
        data_source = f"v4 예약 데이터 (예약자=탑승자 근사, 노쇼 {int(NO_SHOW_RATE*100)}% 보정)"
    else:
        data_source = "운행실적 시트(실측 탑승)"
    payload = {
        "data_source": data_source,
        "weeks_analyzed": len(ALL_WEEKS),
        "policy": {"promote_weeks": PROMOTE_WEEKS, "demote_weeks": DEMOTE_WEEKS,
                   "retire_weeks": RETIRE_WEEKS, "conditional_min": CONDITIONAL_MIN,
                   "demote_load": DEMOTE_LOAD, "retire_resv": RETIRE_RESV},
        "proposals": proposals,
        "threshold_relearn": threshold,
    }
    resp = client.chat.completions.create(
        model=MODEL, temperature=0.2,
        messages=[{"role": "system", "content": REPORT_SYSTEM},
                  {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}],
    )
    return resp.choices[0].message.content

report = build_report()
print(report)

In [ ]:
# [셀 8] (선택) 운행실적 1건 기록 — 운영자가 운행 후 실제 탑승 인원 입력용
def log_operation(op_date, direction, ktx_time, actual_riders,
                  reservations=None, dispatched=True, note=""):
    """운행실적 시트에 1행 추가. op_date='YYYY-MM-DD', direction='to_station'|'to_campus'."""
    wd = datetime.strptime(op_date, "%Y-%m-%d").weekday()
    key = slot_key(direction, wd, ktx_time.strip())
    meta = SLOT_CATALOG.get(key)
    slot = meta["slot"] if meta else "(미등록)"
    service = meta["grade"] if meta else "unknown"
    if reservations is None:
        reservations = count = 0
    ops_ws.append_row(
        [op_date, direction, ktx_time.strip(), slot, service,
         reservations, actual_riders, "TRUE" if dispatched else "FALSE", note],
        value_input_option="RAW")
    print(f"기록 완료: {op_date} {slot} 탑승 {actual_riders}명")

# 예시:
# log_operation("2026-05-29", "to_station", "13:58", actual_riders=9, reservations=9)
print("log_operation() 준비 완료 — 실제 운행 후 호출해 실적을 쌓으세요.")

In [ ]:
# [셀 9] 웹 대시보드 (Gradio) — "분석 실행" 버튼으로 제안서 생성
# 앞 셀(2~7)이 정의한 함수/상수를 재사용한다. 먼저 위 셀들을 모두 실행해 둘 것.
import gradio as gr

GRADE_KR = {"fixed": "고정", "conditional": "조건부"}

def _refresh_data():
    """최신 시트 데이터를 다시 읽어 전역 집계를 갱신 (버튼 누를 때마다 최신화)."""
    global resv_agg, ops_agg, ALL_WEEKS, DATA_IS_DEMO
    resv_agg = load_reservations()
    ops_agg2 = load_operations()
    DATA_IS_DEMO = False

    if RIDERS_FROM_RESERVATIONS and len(ops_agg2) < len(SLOT_CATALOG):
        synth = {}
        for (wk, key), resvN in resv_agg.items():
            meta = SLOT_CATALOG.get(key)
            if meta is None:
                continue
            dispatched = (meta["grade"] == "fixed") or (resvN >= CONDITIONAL_MIN)
            riders = round(resvN * (1 - NO_SHOW_RATE)) if dispatched else 0
            synth[(wk, key)] = {"reservations": resvN, "actual_riders": riders,
                                "dispatched": dispatched}
        if synth:
            ops_agg2 = synth

    if USE_DEMO_DATA and (len(ops_agg2) < len(SLOT_CATALOG)):
        _, ops_agg2 = make_demo()
        DATA_IS_DEMO = True

    ops_agg = ops_agg2
    ALL_WEEKS = sorted({wk for (wk, _key) in ops_agg.keys()})

def _source_label():
    if DATA_IS_DEMO:
        return f"⚠️ 데모 데이터 (실데이터 아님) · 주차 {len(ALL_WEEKS)}개"
    if RIDERS_FROM_RESERVATIONS:
        return f"✅ v4 예약 데이터 (예약자=탑승자 근사, 노쇼 {int(NO_SHOW_RATE*100)}%) · 주차 {len(ALL_WEEKS)}개"
    return f"✅ 운행실적 실측 · 주차 {len(ALL_WEEKS)}개"

def _proposals_table(props):
    """제안 결과를 표(list of rows)로."""
    rows = []
    for p in props:
        rows.append([p["slot"], GRADE_KR.get(p["grade"], p["grade"]),
                     f'{p["avg_resv"]:.1f}', f'{p["avg_rider"]:.1f}',
                     p["action"], p["reason"]])
    return rows

def run_analysis():
    """버튼 콜백: 데이터 갱신 → 판정 → 표 + N* + GPT 제안서."""
    _refresh_data()
    props = reclassify()
    th = relearn_threshold()

    # 전역에 최신 결과 반영(제안서 작성 함수가 참조)
    global proposals, threshold
    proposals, threshold = props, th

    table = _proposals_table(props)
    nstar_md = (rf"**N\* 재학습 결과** — 편익/인 {th['benefit_per_person']:,}원 "
                + rf"→ 손익분기 인원 **N\* = {th['nstar']}명**"
                + f"\n\n_{th['note']}_")
    try:
        report = build_report()
    except Exception as e:
        report = f"(제안서 생성 실패: {e}\n— OPENAI_API_KEY Secret 확인 필요)"
    changes = [p for p in props if p["action"] != "유지"]
    summary = (f"변경 권고 {len(changes)}건"
               + (": " + ", ".join(f'{c["slot"]}→{c["action"]}' for c in changes)
                  if changes else " (이번 주 변경 권고 없음)"))
    return _source_label(), summary, table, nstar_md, report

with gr.Blocks(title="UNIST 운영 최적화 에이전트") as demo:
    gr.Markdown("## 🚍 UNIST ↔ 울산역 운영 최적화 에이전트\n"
                r"v4 예약 데이터를 분석해 **슬롯 등급 재편성 · N\* 재학습**을 제안합니다. "
                "(예약자=탑승자 근사 · 제안은 **사람 승인 후** 적용)")
    with gr.Row():
        run_btn = gr.Button("📊 주간 분석 실행", variant="primary")
    src_out = gr.Textbox(label="데이터 소스", interactive=False)
    sum_out = gr.Textbox(label="요약", interactive=False)
    table_out = gr.Dataframe(
        headers=["슬롯", "현재 등급", "평균 예약", "평균 탑승", "제안", "근거"],
        datatype=["str", "str", "str", "str", "str", "str"],
        label="슬롯별 판정", wrap=True, interactive=False)
    nstar_out = gr.Markdown()
    report_out = gr.Textbox(label="📝 운영 재조정 제안서 (GPT 서술)", lines=14, interactive=False)

    run_btn.click(run_analysis, inputs=None,
                  outputs=[src_out, sum_out, table_out, nstar_out, report_out])

demo.launch(share=True, debug=True)